In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.12 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Co₂SiO₄ on D20 at 2.3 K

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the magnetic structure of the olivine Co₂SiO₄ to constant-wavelength neutron powder data
from D20 at ILL, measured at 2.3 K, below its ordering temperature. The cobalt moments order with
k = 0 in the magnetic space group Pnma.1: Co1 may point in any direction, Co2 only along b.

The project is the catalogue case `pd-neut-cwl_cosio-d20_2k`, the one the command line and the
app's Examples open. Its two Co sites share one displacement, and the two FCJ asymmetry terms are
tied, from the start.

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-cwl_cosio-d20_2k', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['cosio']
experiment = project.experiments['d20']
project.display.constraints()

Data 'pd-neut-cwl_cosio-d20_2k' is at data/pd-neut-cwl_cosio-d20_2k-84d4534215e5/pd-neut-cwl_cosio-d20_2k


View(html=<78,814 characters>, height=264)

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('d20')

View(html=<308,745 characters>, height=480)

## Perform the analysis

Everything starts fixed, and five stages each free one more group, keeping the earlier ones free:
the scale, cell and moments, then the background, the atoms, the peak widths and last the peak
asymmetry.

### Scale, 2θ zero, cell and magnetic moments

The first stage frees the phase scale, the 2θ zero shift, the three cell lengths and the magnetic
model: the moment components the magnetic space group allows (Co1 in any direction, Co2 only
along b) and the j2 term of the Co²⁺ form factor, which sets how the magnetic intensities fall
with angle.

In [6]:
experiment.linked_structures['cosio'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True
structure.atom_site_moment['Co1'].crystalaxis_x.free = True
structure.atom_site_moment['Co1'].crystalaxis_y.free = True
structure.atom_site_moment['Co1'].crystalaxis_z.free = True
structure.atom_site_moment['Co2'].crystalaxis_y.free = True
structure.magnetic_form_factor['Co'].j2_coefficient.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cosio,cell,,length_a,10.3081,,0,30,Å
2,cosio,cell,,length_b,6.00219,,0,30,Å
3,cosio,cell,,length_c,4.78739,,0,30,Å
4,cosio,atom_site_moment,Co1,crystalaxis_x,2,,-inf,inf,μB
5,cosio,atom_site_moment,Co1,crystalaxis_y,2,,-inf,inf,μB
6,cosio,atom_site_moment,Co1,crystalaxis_z,2,,-inf,inf,μB
7,cosio,atom_site_moment,Co2,crystalaxis_y,2,,-inf,inf,μB
8,cosio,magnetic_form_factor,Co,j2_coefficient,0,,-inf,inf,
9,d20,instrument,,calib_twotheta_offset,0.29433,,-inf,inf,deg
10,d20,linked_structure,cosio,scale,1.81723,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 10 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       11.38        35.86
      1      0.029      5.97         9.88   72.5% ↓
      2      0.034      5.97         9.88   
      3      0.040      5.97         9.88   
      4      0.046      5.97         9.88   
  done · converged · 4 iters · reduced χ² 9.88 · Rwp 5.97 % · 0.05 s


View(html=<93,283 characters>, height=376)

In [9]:
project.display.pattern('d20')

View(html=<309,619 characters>, height=480)

### Background

The second stage frees the sixteen background points, which follow the level between the peaks.

In [10]:
for point in experiment.background:
    point.intensity.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 26 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.97         9.99
      1      0.034      5.65         8.93   10.6% ↓
      2      0.040      5.65         8.93   
      3      0.047      5.65         8.93   
  done · converged · 3 iters · reduced χ² 8.93 · Rwp 5.65 % · 0.05 s


View(html=<114,045 characters>, height=376)

In [12]:
project.display.pattern('d20')

View(html=<309,452 characters>, height=480)

### Atomic displacements and positions

The third stage frees the atoms: each site's isotropic displacement (Co2's follows Co1's by the
project's constraint) and the coordinates the symmetry leaves free. Co1 sits on a centre of
symmetry, so it has none; the other sites lie on the mirror plane, except O3.

In [13]:
for site in structure.atom_sites:
    site.adp_iso.free = not site.adp_iso.user_constrained
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 42 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.65         9.03
      1      0.035      5.60         8.87   1.8% ↓
      2      0.041      5.59         8.87   
      3      0.047      5.59         8.87   
      4      0.052      5.59         8.86   
      5      0.058      5.59         8.86   
      6      0.065      5.59         8.86   
  done · converged · 6 iters · reduced χ² 8.86 · Rwp 5.59 % · 0.07 s


View(html=<133,576 characters>, height=376)

In [15]:
project.display.pattern('d20')

View(html=<309,600 characters>, height=480)

### Peak widths

The fourth stage frees the peak profile: the Gaussian widths U, V and W and the Lorentzian Y.

In [16]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 46 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.59         8.89
      1      0.043      5.58         8.86   0.3% ↓
      2      0.050      5.58         8.86   
      3      0.057      5.58         8.86   
      4      0.063      5.58         8.86   
      5      0.069      5.58         8.86   
      6      0.076      5.58         8.86   
      7      0.082      5.58         8.86   
      8      0.088      5.58         8.86   
      9      0.097      5.58         8.86   
  done · converged · 9 iters · reduced χ² 8.86 · Rwp 5.58 % · 0.10 s


View(html=<136,436 characters>, height=376)

In [18]:
project.display.pattern('d20')

View(html=<309,647 characters>, height=480)

### Peak asymmetry

The last stage frees the asymmetry: the low-angle peaks lean towards low angle. One FCJ term is
freed; the other follows it by the project's constraint.

In [19]:
experiment.peak.asym_fcj_1.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 47 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.58         8.87
      1      0.252      5.51         8.62   2.8% ↓
      2      0.258      5.44         8.41   2.4% ↓
      3      0.265      5.36         8.16   3.0% ↓
      4      0.272      5.21         7.71   5.5% ↓
      5      0.279      5.06         7.28   5.6% ↓
      6      0.286      4.74         6.39   12.1% ↓
      7      0.294      4.72         6.33   1.0% ↓
      8      0.301      4.70         6.27   1.0% ↓
      9      0.310      4.69         6.26   0.1% ↓
     10      0.319      4.69         6.26   
     11      0.330      4.69         6.26   
     12      0.340      4.69         6.26   
     13      0.349      4.69         6.25   0.1% ↓
     14      0.360      4.69         6.25   
     15      0.371      4.69         6.25   
     16      0.381      4.69         6.25   
     17      0.391      4.69         6.25   


View(html=<142,632 characters>, height=376)

In [21]:
project.display.pattern('d20')

View(html=<309,911 characters>, height=480)

In [22]:
project.display.fit.correlations()

,1,2,3,4,5
1 d20.instrument.calib_twotheta_offset,,,,,
2 d20.peak.broad_gauss_u,-0.24,,,,
3 d20.peak.broad_gauss_v,0.32,-0.94,,,
4 d20.peak.broad_gauss_w,-0.29,0.83,-0.92,,
5 d20.peak.asym_fcj_1,0.78,-0.30,0.40,-0.37,


In [23]:
project.display.structure('cosio')

View(html=<340,636 characters>, height=660)

## Save the project

In [24]:
project.save_as('pd-neut-cwl_cosio-d20_2k')
project.report.save_html('pd-neut-cwl_cosio-d20_2k.html')

Saving project to 'pd-neut-cwl_cosio-d20_2k'
Saving report to 'pd-neut-cwl_cosio-d20_2k.html'
